# Pose Estimation Model Training in Google Colab

## 1. Setup Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms.functional as TF
from torchvision import transforms
import numpy as np
import os, json
import matplotlib.pyplot as plt

## 2. Dataset Definition

In [ ]:
def generate_heatmaps(keypoints, output_res, sigma=2):
    heatmaps = np.zeros((keypoints.shape[0], output_res[0], output_res[1]), dtype=np.float32)
    for i, (x, y) in enumerate(keypoints):
        if x < 0 or x >= output_res[1] or y < 0 or y >= output_res[0]:
            continue

        xx, yy = np.meshgrid(np.arange(output_res[1]), np.arange(output_res[0]))
        heatmap = np.exp(-((xx - x)**2 + (yy - y)**2) / (2 * sigma**2))
        heatmaps[i] = heatmap
    
    return torch.from_numpy(heatmaps)

class PoseDataset(Dataset):
    def __init__(self, data_dir, num_keypoints=33, output_res=(240, 240)):
        self.data_dir = data_dir
        self.depth_dir = os.path.join(data_dir, 'depth')
        self.confidence_dir = os.path.join(data_dir, 'confidence')
        self.pose_dir = os.path.join(data_dir, 'pose')
        
        self.file_list = [f.split('.')[0] for f in os.listdir(self.depth_dir)]
        self.num_keypoints = num_keypoints
        self.output_res = output_res

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        filename = self.file_list[idx]

        depth_path = os.path.join(self.depth_dir, f"{filename}.npy")
        confidence_path = os.path.join(self.confidence_dir, f"{filename}.npy")

        depth_map = np.load(depth_path)
        confidence_map = np.load(confidence_path)

        pose_path = os.path.join(self.pose_dir, f"{filename}.json")
        with open(pose_path, 'r') as f:
            pose_data = json.load(f)
        keypoints_2d = np.array(pose_data['transformed_points'])

        depth_map = depth_map / 4000.0
        confidence_map = confidence_map / 255.0
        
        input_tensor = torch.from_numpy(np.stack([depth_map, confidence_map], axis=0)).float()

        _, h, w = input_tensor.shape
        pad_left = (self.output_res[1] - w) // 2
        pad_right = self.output_res[1] - w - pad_left
        pad_top = (self.output_res[0] - h) // 2
        pad_bottom = self.output_res[0] - h - pad_top
        padding = (pad_left, pad_top, pad_right, pad_bottom) 
        input_tensor = transforms.functional.pad(input_tensor, padding)

        keypoints_2d[:, 0] += pad_left
        keypoints_2d[:, 1] += pad_top

        target_heatmaps = generate_heatmaps(keypoints_2d, self.output_res, sigma=2)

        return input_tensor, target_heatmaps

## 3. Model Definition (PoseUNet)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)
    
class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = nn.functional.pad(x1, [diffX // 2, diffX - diffX // 2,
                                    diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

class PoseUNet(nn.Module):
    def __init__(self, n_channels, n_keypoints, bilinear=True):
        super(PoseUNet, self).__init__()
        self.n_channels = n_channels
        self.n_keypoints = n_keypoints
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_keypoints)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return torch.sigmoid(logits)

## 4. Training Configuration

In [ ]:
# --- Hyperparameters ---
LEARNING_RATE = 1e-4
BATCH_SIZE = 4 
EPOCHS = 50
# IMPORTANT: Update this path to point to your dataset in Google Drive
DATA_DIR = "/content/drive/MyDrive/your_data_folder/data" 
NUM_KEYPOINTS = 33
INPUT_CHANNELS = 2
IMAGE_RESOLUTION = (240, 240)
VAL_SPLIT = 0.2
RANDOM_SEED = 123

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Using device: {device}')

## 5. Training Loop

In [ ]:
model = PoseUNet(n_channels=INPUT_CHANNELS, n_keypoints=NUM_KEYPOINTS).to(device)
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

full_dataset = PoseDataset(data_dir=DATA_DIR, output_res=IMAGE_RESOLUTION)
print(f"Total number of samples: {len(full_dataset)}")
dataset_size = len(full_dataset)
val_size = int(dataset_size * VAL_SPLIT)
train_size = dataset_size - val_size
print(f"Training size: {train_size}, Validation size: {val_size}")
generator = torch.Generator().manual_seed(RANDOM_SEED)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device)
        targets = targets.to(device)

        predictions = model(data)
        loss = loss_function(predictions, targets)
        total_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS}, Training Loss: {avg_loss:.6f}")

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for data, targets in val_loader:
            data = data.to(device)
            targets = targets.to(device)
            predictions = model(data)
            val_loss = loss_function(predictions, targets)
            total_val_loss += val_loss.item()
    
    avg_val_loss = total_val_loss / len(val_loader)
    print(f"Epoch {epoch+1}/{EPOCHS}, Validation Loss: {avg_val_loss:.6f}")

torch.save(model.state_dict(), "/content/drive/MyDrive/pose_unet_model.pth")
print("Model saved to Google Drive!")

## 6. Visualize Dataset (Optional)

In [ ]:
def visualize_ground_truth(loader, num_samples):
    for i, (inputs, targets) in enumerate(loader):
        if i * loader.batch_size >= num_samples:
            break

        inputs = inputs.cpu().numpy()
        targets = targets.cpu().numpy()

        for j in range(inputs.shape[0]):
            sample_idx = i * loader.batch_size + j
            if sample_idx >= num_samples:
                break

            depth_map = inputs[j, 0, :, :]
            confidence_map = inputs[j, 1, :, :]
            summed_heatmaps = np.sum(targets[j], axis=0)
            
            fig, axs = plt.subplots(1, 4, figsize=(20, 5))
            fig.suptitle(f'Sample #{sample_idx}', fontsize=16)

            im1 = axs[0].imshow(depth_map, cmap='viridis')
            axs[0].set_title('Padded Depth Map (Input Ch 1)')
            axs[0].axis('off')
            fig.colorbar(im1, ax=axs[0], fraction=0.046, pad=0.04)

            im2 = axs[1].imshow(confidence_map, cmap='magma')
            axs[1].set_title('Padded Confidence Map (Input Ch 2)')
            axs[1].axis('off')
            fig.colorbar(im2, ax=axs[1], fraction=0.046, pad=0.04)

            im3 = axs[2].imshow(summed_heatmaps, cmap='hot')
            axs[2].set_title('Summed Heatmaps (Ground Truth)')
            axs[2].axis('off')
            fig.colorbar(im3, ax=axs[2], fraction=0.046, pad=0.04)

            axs[3].imshow(depth_map, cmap='viridis')
            axs[3].imshow(summed_heatmaps, cmap='hot', alpha=0.6)
            axs[3].set_title('Overlay: Heatmaps on Depth')
            axs[3].axis('off')

            plt.tight_layout()
            plt.show()

vis_loader = DataLoader(dataset=val_dataset, batch_size=4, shuffle=True)
visualize_ground_truth(vis_loader, num_samples=8)